# Classes and domain modeling: objects with scientific meaning

Represent mathematical, physical, chemical, and industrial-engineering objects so that invalid
states are rejected early and trustworthy functions can compute with them.

**Lecture 2 · Python Foundations II · CMOR 438 / INDE 577**

## Python focus: the sciences are teaching contexts

This is a **Python classes and data-modeling lesson**. The domain examples make each design choice
concrete; they are not substitutes for courses in physics, chemistry, or queueing theory.

| Python learning target | Domain object used to teach it |
| --- | --- |
| class, instance, attribute, method, and `self` | a vector in the plane |
| dataclass, equality, immutability, and validation | a trustworthy `Vector2D` |
| composition | a physical state containing position and velocity vectors |
| quantities, units, and boundary checks | a chemical species and sample |
| object-in/object-out functions and custom exceptions | an M/M/1 service station |
| controlled mutable state | an online running mean |

The transferable question is: **What must always be true for this object to mean what its name
claims?**

## How to use this notebook

**Estimated time:** 60 minutes of core instruction, plus 40 minutes of practice and extension.

**Prerequisite:** Lecture 2 notebook 00 on functions, NumPy-style docstrings, type hints,
validation, and exceptions.

Follow **Core** during class. Before constructing an object, predict its attributes and its valid
domain. Before calling a function, identify the input model, return model, units, assumptions, and
possible failure. Run from top to bottom in the **Rice DSM** kernel.

The complete notebook is deliberately richer than a live route. In class, prioritize `Vector2D`,
`ParticleState`, the queue worked example, and guided practice; use chemistry and extensions for
review or homework.

## Learning objectives

By the end of this notebook, you should be able to:

- explain the relationships among a class, an instance, attributes, methods, and `self`;
- choose among a plain value, dictionary, dataclass, and stateful class;
- state and enforce a domain object's invariants at its construction boundary;
- combine NumPy-style docstrings, type hints, validation, and actionable exceptions;
- distinguish value equality from object identity and shallow from deep immutability;
- use composition to build a model from smaller meaningful objects;
- justify whether a computation belongs in a method or a separate function;
- design pure, typed functions that consume and return domain models; and
- test valid behavior, invalid inputs, units, and scientific assumptions.

## Why this matters in industry

Real projects pass objects between data ingestion, simulation, feature engineering, training,
evaluation, APIs, and reports. A bare tuple such as `(2.0, 5.0)` does not reveal whether its values
are coordinates, rates, concentrations, or model scores. A dictionary is more descriptive, but it
may still omit a key, mix units, or contain an impossible value.

A good domain model creates a **semantic boundary**. After construction succeeds, downstream code
may rely on named invariants: mass is positive, a vector has two finite components, and a service
station has positive capacity. That reduces defensive checks, makes interfaces readable, and turns
scientific assumptions into testable code.

Classes are not automatically better. If a value has no enduring identity, invariant, or behavior,
a number, tuple, or dictionary may be clearer. The cost of a class is another interface that a team
must document, test, and maintain.

## The modeling rhythm

We will repeatedly move through this chain:

```text
domain concept
    → named attributes and units
    → invariants checked at construction
    → valid object
    → typed function
    → computed object or quantity
    → assertion and interpretation
```

An **invariant** is a condition promised to remain true for every valid instance. Rejecting an
invalid object at the boundary is usually safer than letting an error surface much later inside a
calculation.

## Professional practice: two kinds of correctness

| Data scientist asks | Software engineer asks |
| --- | --- |
| What scientific object is represented? | What public interface represents it? |
| What are the units and valid domain? | Where are those invariants checked? |
| Which assumptions justify the calculation? | Are failures explicit and testable? |
| Is this observed state or a derived result? | Should the operation be a method or function? |
| Could mutation invalidate an analysis? | Should this be an immutable value object? |
| Does equality have domain meaning? | Are `__eq__` and hashing behavior safe? |

Professional code needs both. A beautifully documented class with the wrong units is scientifically
wrong; a correct equation behind an ambiguous, mutation-prone interface is operationally fragile.

## 1. The smallest useful class

A **class** defines how a kind of object is constructed and what behavior it exposes. Calling the
class creates an **instance**. Instance attributes store that particular object's state.

`__init__` initializes a new instance. The conventional first parameter `self` refers to the
instance receiving a method call; `self` is a convention, not a reserved keyword. The explicit name
makes it clear whether code is reading local state or instance state.

In [ ]:
class PlainVector2D:
    """Represent a two-dimensional vector without automatic validation.

    Parameters
    ----------
    horizontal : float
        Horizontal component.
    vertical : float
        Vertical component.
    """

    def __init__(self, horizontal: float, vertical: float) -> None:
        self.horizontal = horizontal
        self.vertical = vertical

    def as_tuple(self) -> tuple[float, float]:
        """Return the components in horizontal-vertical order."""

        return (self.horizontal, self.vertical)


wind_velocity = PlainVector2D(3.0, 4.0)

assert wind_velocity.horizontal == 3.0
assert wind_velocity.as_tuple() == (3.0, 4.0)

### Trace a bound method

When Python evaluates `wind_velocity.as_tuple()`, it finds the function on the class and binds the
instance to it. Conceptually, the call is equivalent to
`PlainVector2D.as_tuple(wind_velocity)`.

**Predict before running:** Which object will appear as `__self__`? Why does the explicit class call
need an argument while the bound call does not?

In [ ]:
bound_method = wind_velocity.as_tuple

print("bound to:", bound_method.__self__)
print("same result:", PlainVector2D.as_tuple(wind_velocity))

assert bound_method.__self__ is wind_velocity
assert PlainVector2D.as_tuple(wind_velocity) == bound_method()

### Type hints describe intent; Python does not enforce them

`horizontal: float` helps readers, editors, and static type checkers. At runtime, however, the plain
class will accept a string. Construction succeeds even though later arithmetic may fail. This is a
critical distinction:

- **type hints** describe the intended interface;
- a **static type checker** analyzes code without running it;
- **runtime validation** protects boundaries when values actually arrive.

Do not deliberately compute with the invalid instance below; its existence is the lesson.

In [ ]:
invalid_plain_vector = PlainVector2D("east", 4.0)  # accepted at runtime

assert invalid_plain_vector.horizontal == "east"
assert PlainVector2D.__annotations__ == {}
assert PlainVector2D.__init__.__annotations__["horizontal"] is float

## 2. Dataclasses for value objects

The plain class repeats common plumbing. A standard-library `@dataclass` can generate construction,
representation, and value-based equality from annotated fields. It does **not** decide the scientific
meaning or validate inputs for us.

Our vector is a **value object**: two vectors with the same components should compare equal, and
changing a vector after it has been used in a calculation would be surprising. We therefore choose
`frozen=True`. We choose `slots=True` to declare a fixed attribute layout and prevent accidental new
attributes; this is useful, not a universal requirement.

In [ ]:
from dataclasses import FrozenInstanceError, asdict, dataclass, field
from math import cos, hypot, isfinite, sin
from numbers import Real
from typing import Self

### Validate once, near the boundary

Several models need finite real numbers. A private helper centralizes that policy. It rejects
Booleans because `bool` is technically a subclass of `int` but rarely represents a scientific
quantity. It converts accepted integers and floats to a consistent `float` representation.

The exception message names the parameter, received value, and required domain. That is far more
actionable than a later message such as “unsupported operand type.”

In [ ]:
def _finite_float(value: object, *, parameter: str) -> float:
    """Return `value` as a finite float.

    Parameters
    ----------
    value : object
        Candidate numeric value.
    parameter : str
        Public parameter name used in an error message.

    Returns
    -------
    float
        Finite floating-point representation of `value`.

    Raises
    ------
    TypeError
        If `value` is not a real number or is a Boolean.
    ValueError
        If `value` is infinite or not a number.
    """

    if isinstance(value, bool) or not isinstance(value, Real):
        raise TypeError(
            f"{parameter} must be a real number; received {value!r}"
        )
    result = float(value)
    if not isfinite(result):
        raise ValueError(f"{parameter} must be finite; received {value!r}")
    return result

In [ ]:
@dataclass(frozen=True, slots=True)
class Vector2D:
    """Represent a finite vector in the Euclidean plane.

    Parameters
    ----------
    horizontal : float
        Horizontal component in the units stated by the caller.
    vertical : float
        Vertical component in the same units as `horizontal`.

    Raises
    ------
    TypeError
        If either component is not a real number.
    ValueError
        If either component is not finite.

    Notes
    -----
    A vector stores components, not their physical units. A surrounding domain
    model must state whether these components mean meters, meters per second, or
    another compatible unit.
    """

    horizontal: float
    vertical: float

    def __post_init__(self) -> None:
        """Validate and normalize components after generated initialization."""

        object.__setattr__(
            self,
            "horizontal",
            _finite_float(self.horizontal, parameter="horizontal"),
        )
        object.__setattr__(
            self,
            "vertical",
            _finite_float(self.vertical, parameter="vertical"),
        )

    def as_tuple(self) -> tuple[float, float]:
        """Return components in horizontal-vertical order."""

        return (self.horizontal, self.vertical)

    @classmethod
    def from_polar(cls, radius: object, angle_radians: object) -> Self:
        """Construct a vector from polar coordinates.

        Parameters
        ----------
        radius : object
            Nonnegative radial coordinate.
        angle_radians : object
            Finite angle measured counterclockwise in radians.

        Returns
        -------
        Vector2D
            Vector with the corresponding Cartesian components.

        Raises
        ------
        TypeError
            If either argument is not a real number.
        ValueError
            If either argument is not finite or `radius` is negative.
        """

        finite_radius = _finite_float(radius, parameter="radius")
        finite_angle = _finite_float(angle_radians, parameter="angle_radians")
        if finite_radius < 0.0:
            raise ValueError(f"radius must be nonnegative; received {radius!r}")
        return cls(
            finite_radius * cos(finite_angle),
            finite_radius * sin(finite_angle),
        )

`__post_init__` runs after the generated initializer. Because the class is frozen, normalization
uses `object.__setattr__` only during construction. This is a controlled implementation detail, not
permission to mutate instances later.

The class method is an **alternate constructor**: constructing a `Vector2D` is behavior naturally
owned by the class. Its return annotation `Self` means “an instance of the class on which this method
was called.”

In [ ]:
first_vector = Vector2D(3, 4)
equivalent_vector = Vector2D(3.0, 4.0)
unit_horizontal = Vector2D.from_polar(1.0, 0.0)

print(first_vector)
print("value equality:", first_vector == equivalent_vector)
print("object identity:", first_vector is equivalent_vector)

assert first_vector == equivalent_vector
assert first_vector is not equivalent_vector
assert first_vector.as_tuple() == (3.0, 4.0)
assert unit_horizontal == Vector2D(1.0, 0.0)
assert len({first_vector, equivalent_vector}) == 1

### Equality, identity, and hashing

- `a == b` asks whether two objects have equal **values** according to their type.
- `a is b` asks whether two names refer to the exact same object.
- A frozen dataclass whose fields are hashable can usually be used in a set or as a dictionary key.

Never replace value comparison with `is`. Identity is appropriate for a unique sentinel such as
`None`, not for scientific values. Hashable objects must not change in ways that alter equality;
that is one reason immutable value objects work well as keys.

In [ ]:
try:
    first_vector.horizontal = 99.0
except FrozenInstanceError as error:
    print(type(error).__name__ + ":", error)

assert first_vector.horizontal == 3.0

## 3. Functions that compute with mathematical objects

The vector owns its representation and construction rules. Dot products and magnitudes are
algorithms that consume vectors, so separate pure functions keep representation and computation
loosely coupled. A later implementation could optimize the algorithm without changing the model.

There is no universal prohibition against methods such as `vector.magnitude()`. The design question
is whether the behavior is intrinsic, discoverable, and stable enough to belong to the object's
small public interface.

In [ ]:
def dot_product(left: Vector2D, right: Vector2D) -> float:
    """Compute the Euclidean dot product of two vectors.

    Parameters
    ----------
    left : Vector2D
        First vector.
    right : Vector2D
        Second vector in units compatible with `left`.

    Returns
    -------
    float
        Sum of componentwise products.
    """

    return (
        left.horizontal * right.horizontal
        + left.vertical * right.vertical
    )


def magnitude(vector: Vector2D) -> float:
    """Compute the Euclidean magnitude of a vector.

    Parameters
    ----------
    vector : Vector2D
        Vector whose magnitude is required.

    Returns
    -------
    float
        Nonnegative magnitude in the vector's component units.
    """

    return hypot(vector.horizontal, vector.vertical)


assert dot_product(Vector2D(1, 0), Vector2D(0, 1)) == 0.0
assert magnitude(first_vector) == 5.0

### Method or function?

Use this course heuristic, then document exceptions in code review:

| Prefer a method when... | Prefer a function when... |
| --- | --- |
| behavior is intrinsic to one object's abstraction | an algorithm combines several objects |
| it establishes or safely changes object state | it should be pure and independently testable |
| it is a natural constructor or representation | multiple algorithms may operate on the model |
| users will look for it on the object | policy belongs to an analysis layer |

Avoid “god objects” that ingest files, clean data, train models, draw plots, and write reports. A
small model plus focused functions is often easier to test and reuse.

## 4. Physics: build larger models by composition

**Composition** means that one object contains other meaningful objects. A particle state has a
position vector and a velocity vector; it does not inherit from `Vector2D`. This models a “has-a”
relationship and lets `Vector2D` keep one clear responsibility.

We state an explicit unit convention: mass in kilograms, position in meters, and velocity in meters
per second. Production software may use a dedicated units library; here, names and documentation
make the convention visible without third-party packages.

In [ ]:
@dataclass(frozen=True, slots=True)
class ParticleState:
    """Represent the planar state of a classical point particle.

    Parameters
    ----------
    label : str
        Nonblank identifier for the particle.
    mass_kg : float
        Strictly positive mass in kilograms.
    position_m : Vector2D
        Planar position in meters.
    velocity_m_per_s : Vector2D
        Planar velocity in meters per second.

    Raises
    ------
    TypeError
        If a field has the wrong runtime type.
    ValueError
        If `label` is blank or `mass_kg` is not finite and positive.
    """

    label: str
    mass_kg: float
    position_m: Vector2D
    velocity_m_per_s: Vector2D

    def __post_init__(self) -> None:
        if not isinstance(self.label, str):
            raise TypeError(f"label must be a string; received {self.label!r}")
        normalized_label = " ".join(self.label.split())
        if not normalized_label:
            raise ValueError("label must contain at least one non-whitespace character")
        if not isinstance(self.position_m, Vector2D):
            raise TypeError("position_m must be a Vector2D instance")
        if not isinstance(self.velocity_m_per_s, Vector2D):
            raise TypeError("velocity_m_per_s must be a Vector2D instance")
        finite_mass = _finite_float(self.mass_kg, parameter="mass_kg")
        if finite_mass <= 0.0:
            raise ValueError(f"mass_kg must be positive; received {self.mass_kg!r}")
        object.__setattr__(self, "label", normalized_label)
        object.__setattr__(self, "mass_kg", finite_mass)

In [ ]:
def kinetic_energy(particle: ParticleState) -> float:
    """Compute classical translational kinetic energy.

    Parameters
    ----------
    particle : ParticleState
        Particle whose mass is in kilograms and velocity is in meters per second.

    Returns
    -------
    float
        Kinetic energy in joules.
    """

    speed_squared = dot_product(
        particle.velocity_m_per_s,
        particle.velocity_m_per_s,
    )
    return 0.5 * particle.mass_kg * speed_squared


def momentum(particle: ParticleState) -> Vector2D:
    """Compute classical linear momentum.

    Parameters
    ----------
    particle : ParticleState
        Particle with SI mass and velocity components.

    Returns
    -------
    Vector2D
        Momentum components in kilogram-meters per second.
    """

    return Vector2D(
        particle.mass_kg * particle.velocity_m_per_s.horizontal,
        particle.mass_kg * particle.velocity_m_per_s.vertical,
    )


probe = ParticleState(
    label="  probe   A ",
    mass_kg=2.0,
    position_m=Vector2D(10.0, -3.0),
    velocity_m_per_s=Vector2D(3.0, 4.0),
)

assert probe.label == "probe A"
assert kinetic_energy(probe) == 25.0
assert momentum(probe) == Vector2D(6.0, 8.0)

The functions accept a valid `ParticleState`, not four unrelated numbers. Their signatures expose
the expected model, while docstrings carry semantics that annotations cannot: unit conventions and
the classical-mechanics interpretation.

Notice that `kinetic_energy` reuses `dot_product`. Small domain objects and focused functions compose
into more interesting calculations without producing a deep inheritance hierarchy.

## 5. Chemistry: a model prevents unit ambiguity

A molar mass is not merely a positive number; it belongs to a chemical species and has units. We
will store a formula as a label rather than attempt chemical-formula parsing. That explicit scope
matters: the model represents trusted reference data, not arbitrary chemistry notation.

In [ ]:
@dataclass(frozen=True, slots=True)
class ChemicalSpecies:
    """Represent trusted reference data for a chemical species.

    Parameters
    ----------
    name : str
        Nonblank human-readable name.
    formula : str
        Nonblank formula label; it is stored but not parsed.
    molar_mass_g_per_mol : float
        Strictly positive molar mass in grams per mole.
    charge : int, default=0
        Net electric charge in units of the elementary charge.

    Raises
    ------
    TypeError
        If text or charge fields have incorrect runtime types.
    ValueError
        If text is blank or molar mass is not finite and positive.
    """

    name: str
    formula: str
    molar_mass_g_per_mol: float
    charge: int = 0

    def __post_init__(self) -> None:
        if not isinstance(self.name, str):
            raise TypeError(f"name must be a string; received {self.name!r}")
        if not isinstance(self.formula, str):
            raise TypeError(f"formula must be a string; received {self.formula!r}")
        normalized_name = " ".join(self.name.split())
        normalized_formula = self.formula.strip()
        if not normalized_name:
            raise ValueError("name must contain at least one non-whitespace character")
        if not normalized_formula:
            raise ValueError(
                "formula must contain at least one non-whitespace character"
            )
        if isinstance(self.charge, bool) or not isinstance(self.charge, int):
            raise TypeError(f"charge must be an integer; received {self.charge!r}")
        molar_mass = _finite_float(
            self.molar_mass_g_per_mol,
            parameter="molar_mass_g_per_mol",
        )
        if molar_mass <= 0.0:
            raise ValueError(
                "molar_mass_g_per_mol must be positive; "
                f"received {self.molar_mass_g_per_mol!r}"
            )
        object.__setattr__(self, "name", normalized_name)
        object.__setattr__(self, "formula", normalized_formula)
        object.__setattr__(self, "molar_mass_g_per_mol", molar_mass)

In [ ]:
def moles_from_mass(species: ChemicalSpecies, *, mass_g: object) -> float:
    """Convert a sample mass to amount of substance.

    Parameters
    ----------
    species : ChemicalSpecies
        Species with molar mass in grams per mole.
    mass_g : object
        Nonnegative sample mass in grams.

    Returns
    -------
    float
        Amount of substance in moles.

    Raises
    ------
    TypeError
        If `mass_g` is not a real number.
    ValueError
        If `mass_g` is not finite or is negative.
    """

    finite_mass = _finite_float(mass_g, parameter="mass_g")
    if finite_mass < 0.0:
        raise ValueError(f"mass_g must be nonnegative; received {mass_g!r}")
    return finite_mass / species.molar_mass_g_per_mol


water = ChemicalSpecies("water", "H2O", 18.01528)
amount_mol = moles_from_mass(water, mass_g=36.03056)

print(f"{amount_mol:.3f} mol of {water.name}")
assert abs(amount_mol - 2.0) < 1e-12

The keyword-only name `mass_g` makes the unit visible at the call site. The class guarantees a
positive molar mass; the function owns the separate policy that sample mass may be zero but not
negative. Each boundary checks only the invariant it owns.

This model still cannot prove that `18.01528` matches `H2O`. In production, that relationship would
need a trusted source, provenance, tests, and perhaps a more specialized chemistry library. A class
can enforce coded rules; it cannot manufacture domain truth.

## 6. Worked example: an industrial service station

Suppose jobs arrive at a single processing station at average rate $\lambda$, while the station
serves jobs at average rate $\mu$. Under the **M/M/1** assumptions—Poisson arrivals, independent
exponential service times, one server, a stationary process, and $0 \leq \lambda < \mu$—the
utilization and expected time in the system are

$$
\rho = \frac{\lambda}{\mu},
\qquad
W = \frac{1}{\mu - \lambda}.
$$

The formulas are the domain layer. Our Python question is how to represent inputs and results so
units, assumptions, and instability are hard to ignore.

In [ ]:
class UnstableQueueError(ValueError):
    """Report an arrival rate at or above a station's service capacity."""


@dataclass(frozen=True, slots=True)
class ServiceStation:
    """Represent one processing station for an M/M/1 model.

    Parameters
    ----------
    name : str
        Nonblank station identifier.
    service_rate_per_hour : float
        Strictly positive mean service rate in jobs per hour.

    Raises
    ------
    TypeError
        If `name` is not text or the rate is not real.
    ValueError
        If `name` is blank or the rate is not finite and positive.
    """

    name: str
    service_rate_per_hour: float

    def __post_init__(self) -> None:
        if not isinstance(self.name, str):
            raise TypeError(f"name must be a string; received {self.name!r}")
        normalized_name = " ".join(self.name.split())
        if not normalized_name:
            raise ValueError("name must contain at least one non-whitespace character")
        service_rate = _finite_float(
            self.service_rate_per_hour,
            parameter="service_rate_per_hour",
        )
        if service_rate <= 0.0:
            raise ValueError(
                "service_rate_per_hour must be positive; "
                f"received {self.service_rate_per_hour!r}"
            )
        object.__setattr__(self, "name", normalized_name)
        object.__setattr__(self, "service_rate_per_hour", service_rate)


@dataclass(frozen=True, slots=True)
class QueueMetrics:
    """Store derived steady-state M/M/1 performance metrics.

    Parameters
    ----------
    utilization : float
        Fraction of service capacity demanded, in `[0, 1)`.
    expected_jobs_in_system : float
        Expected number of jobs waiting or in service.
    expected_time_in_system_hours : float
        Expected time from arrival through service, in hours.

    Raises
    ------
    TypeError
        If a metric is not a real number.
    ValueError
        If a metric is not finite or violates its stated domain.
    """

    utilization: float
    expected_jobs_in_system: float
    expected_time_in_system_hours: float

    def __post_init__(self) -> None:
        finite_utilization = _finite_float(
            self.utilization,
            parameter="utilization",
        )
        finite_jobs = _finite_float(
            self.expected_jobs_in_system,
            parameter="expected_jobs_in_system",
        )
        finite_time = _finite_float(
            self.expected_time_in_system_hours,
            parameter="expected_time_in_system_hours",
        )
        if not 0.0 <= finite_utilization < 1.0:
            raise ValueError(
                "utilization must be in [0, 1); "
                f"received {self.utilization!r}"
            )
        if finite_jobs < 0.0:
            raise ValueError(
                "expected_jobs_in_system must be nonnegative; "
                f"received {self.expected_jobs_in_system!r}"
            )
        if finite_time <= 0.0:
            raise ValueError(
                "expected_time_in_system_hours must be positive; "
                f"received {self.expected_time_in_system_hours!r}"
            )
        object.__setattr__(self, "utilization", finite_utilization)
        object.__setattr__(self, "expected_jobs_in_system", finite_jobs)
        object.__setattr__(self, "expected_time_in_system_hours", finite_time)

In [ ]:
def utilization(
    station: ServiceStation,
    *,
    arrival_rate_per_hour: object,
) -> float:
    """Compute demand as a fraction of station capacity.

    Parameters
    ----------
    station : ServiceStation
        Station with service rate measured in jobs per hour.
    arrival_rate_per_hour : object
        Nonnegative arrival rate in jobs per hour.

    Returns
    -------
    float
        Ratio of arrival rate to service rate. Values at or above one are returned;
        they indicate that a steady-state M/M/1 analysis is invalid.

    Raises
    ------
    TypeError
        If `arrival_rate_per_hour` is not a real number.
    ValueError
        If `arrival_rate_per_hour` is not finite or is negative.
    """

    arrival_rate = _finite_float(
        arrival_rate_per_hour,
        parameter="arrival_rate_per_hour",
    )
    if arrival_rate < 0.0:
        raise ValueError(
            "arrival_rate_per_hour must be nonnegative; "
            f"received {arrival_rate_per_hour!r}"
        )
    return arrival_rate / station.service_rate_per_hour


def analyze_mm1(
    station: ServiceStation,
    *,
    arrival_rate_per_hour: object,
) -> QueueMetrics:
    """Compute steady-state metrics for an M/M/1 service station.

    Parameters
    ----------
    station : ServiceStation
        Single-server station with exponential service times.
    arrival_rate_per_hour : object
        Nonnegative Poisson arrival rate in jobs per hour.

    Returns
    -------
    QueueMetrics
        Utilization, expected jobs in the system, and expected system time.

    Raises
    ------
    TypeError
        If `arrival_rate_per_hour` is not a real number.
    ValueError
        If `arrival_rate_per_hour` is not finite or is negative.
    UnstableQueueError
        If the arrival rate is at or above service capacity, so no stationary
        M/M/1 distribution exists.

    Notes
    -----
    Results are meaningful only when the documented M/M/1 assumptions are a
    reasonable model of the observed system. Passing validation is necessary but
    not sufficient for model validity.
    """

    load = utilization(
        station,
        arrival_rate_per_hour=arrival_rate_per_hour,
    )
    if load >= 1.0:
        raise UnstableQueueError(
            f"arrival_rate_per_hour={arrival_rate_per_hour!r} must be below "
            f"{station.name!r} capacity "
            f"({station.service_rate_per_hour} jobs/hour)"
        )
    arrival_rate = float(arrival_rate_per_hour)
    service_margin = station.service_rate_per_hour - arrival_rate
    return QueueMetrics(
        utilization=load,
        expected_jobs_in_system=arrival_rate / service_margin,
        expected_time_in_system_hours=1.0 / service_margin,
    )

In [ ]:
inspection_station = ServiceStation(
    name="final inspection",
    service_rate_per_hour=12.0,
)
inspection_metrics = analyze_mm1(
    inspection_station,
    arrival_rate_per_hour=9.0,
)

print(inspection_metrics)
assert inspection_metrics.utilization == 0.75
assert inspection_metrics.expected_jobs_in_system == 3.0
assert inspection_metrics.expected_time_in_system_hours == 1.0 / 3.0

This function accepts one validated model and returns another. Returning `QueueMetrics` is clearer
than returning an unlabeled three-tuple. The result is immutable because it is a derived snapshot;
recompute it when inputs change.

The `UnstableQueueError` distinguishes a scientifically important model failure from an ordinary
bad numeric input. It subclasses `ValueError`, so callers may catch the precise failure or the
broader built-in category.

In [ ]:
try:
    analyze_mm1(inspection_station, arrival_rate_per_hour=12.5)
except UnstableQueueError as error:
    print(type(error).__name__ + ":", error)

assert utilization(inspection_station, arrival_rate_per_hour=12.5) > 1.0

### A small computational experiment

The formula predicts a nonlinear rise in delay as demand approaches capacity. Creating several
valid input objects and applying the same function makes that behavior inspectable. The calculation
is deterministic; no plotting package is needed to discover the pattern.

**Predict:** Which increase has the larger effect on expected time: 6 to 9 arrivals per hour, or 9
to 11?

In [ ]:
arrival_rates = (6.0, 9.0, 11.0)
capacity_experiment = [
    (
        arrival_rate,
        analyze_mm1(
            inspection_station,
            arrival_rate_per_hour=arrival_rate,
        ),
    )
    for arrival_rate in arrival_rates
]

for arrival_rate, metrics in capacity_experiment:
    print(
        f"arrival={arrival_rate:>4.1f}/h | "
        f"utilization={metrics.utilization:>5.1%} | "
        f"expected time={60 * metrics.expected_time_in_system_hours:>5.1f} min"
    )

expected_times = [
    metrics.expected_time_in_system_hours
    for _, metrics in capacity_experiment
]
assert expected_times[2] - expected_times[1] > expected_times[1] - expected_times[0]

### Interpret before recommending

The output does **not** prove that the real inspection station will have these delays. Before using
the result, investigate the assumptions: Are arrivals approximately Poisson? Are service times
approximately exponential and independent? Is there really one server? Are rates stationary? Does
priority scheduling change the discipline?

Software validation established $0 \leq \lambda < \mu$. Domain validation requires evidence that
M/M/1 is an adequate abstraction. Tests can verify the formula we wrote, but not whether we chose
the right model of reality. **Model validity** is an empirical and scientific question, not merely a
property of well-typed code.

## 7. Not every class should be frozen: controlled state

Vectors and metric snapshots are values. A streaming accumulator represents a process that evolves
as observations arrive, so mutation is its purpose. The class protects that mutable state behind a
small interface: `update` changes it, while read-only properties expose results.

This is different from passing a public dictionary around and allowing any caller to edit internal
counts.

In [ ]:
class RunningMean:
    """Accumulate the arithmetic mean of finite observations.

    Notes
    -----
    The instance is intentionally mutable. Each call to `update` changes its count
    and total. This introductory implementation does not address floating-point
    error or thread safety.
    """

    __slots__ = ("_count", "_total")

    def __init__(self) -> None:
        self._count = 0
        self._total = 0.0

    def update(self, value: object) -> None:
        """Add one finite observation.

        Parameters
        ----------
        value : object
            Real, finite observation.

        Raises
        ------
        TypeError
            If `value` is not a real number.
        ValueError
            If `value` is not finite.
        """

        observation = _finite_float(value, parameter="value")
        self._count += 1
        self._total += observation

    @property
    def count(self) -> int:
        """Number of accepted observations."""

        return self._count

    @property
    def mean(self) -> float:
        """Arithmetic mean of accepted observations.

        Raises
        ------
        RuntimeError
            If no observation has been added.
        """

        if self._count == 0:
            raise RuntimeError("mean is undefined before the first update")
        return self._total / self._count


temperature_mean = RunningMean()
for temperature_c in (20.0, 21.5, 19.0, 21.5):
    temperature_mean.update(temperature_c)

assert temperature_mean.count == 4
assert temperature_mean.mean == 20.5

The leading underscore in `_count` and `_total` is a **convention** meaning non-public; Python does
not make those names truly private. `@property` lets callers read `mean` like an attribute while the
class computes it and may raise a documented error.

Choose mutation deliberately. Stateful objects need tests for operation order, repeated calls,
failure atomicity, and independent instances. They may also need synchronization if shared across
threads—an assumption this teaching implementation explicitly does not satisfy.

### Mutable fields need a factory

A dataclass field such as a list must not be shared accidentally among instances. `field` with
`default_factory=list` calls `list()` separately for each new object. This is the class-field version
of the mutable-default lesson from functions.

In [ ]:
@dataclass(slots=True)
class ExperimentLog:
    """Collect notes for one experiment.

    Parameters
    ----------
    experiment_id : str
        Identifier for the experiment.
    notes : list of str, optional
        Mutable notes owned by this instance.
    """

    experiment_id: str
    notes: list[str] = field(default_factory=list)

    def __post_init__(self) -> None:
        if not isinstance(self.experiment_id, str):
            raise TypeError(
                "experiment_id must be a string; "
                f"received {self.experiment_id!r}"
            )
        normalized_id = self.experiment_id.strip()
        if not normalized_id:
            raise ValueError(
                "experiment_id must contain at least one non-whitespace character"
            )
        object.__setattr__(self, "experiment_id", normalized_id)


first_log = ExperimentLog("trial-001")
second_log = ExperimentLog("trial-002")
first_log.notes.append("sensor stabilized")

assert first_log.notes == ["sensor stabilized"]
assert second_log.notes == []
assert first_log.notes is not second_log.notes

### Frozen is shallow, and slots are a tradeoff

`frozen=True` prevents rebinding dataclass fields; it does not recursively freeze a list stored in a
field. Prefer immutable field types such as tuples inside immutable value objects, or make the
remaining mutability explicit and tested.

`slots=True` can reduce per-instance overhead and reject undeclared attributes. It can also make
some forms of introspection, multiple inheritance, serialization, and weak references more
complicated. Use it when a fixed layout fits the abstraction, not as decoration.

### Serialization is a boundary, not the object itself

`asdict` recursively converts a dataclass to ordinary dictionaries and values. That is convenient
when preparing JSON-compatible data, but type identity and validation behavior are lost. Loading the
dictionary later does not automatically reconstruct a `ParticleState` or its nested vectors.

In [ ]:
probe_record = asdict(probe)

print(probe_record)
assert probe_record["mass_kg"] == 2.0
assert probe_record["velocity_m_per_s"] == {"horizontal": 3.0, "vertical": 4.0}
assert isinstance(probe_record, dict)
assert not isinstance(probe_record, ParticleState)

## Choosing a representation

| Representation | Good fit | Warning sign |
| --- | --- | --- |
| scalar or string | one atomic value | units or meaning are implicit |
| tuple | small fixed anonymous grouping | field order is easy to confuse |
| dictionary | flexible external or evolving records | required keys and invariants are hidden |
| frozen dataclass | validated value with named fields | callers truly need evolving state |
| stateful class | behavior must control a lifecycle | public methods accumulate unrelated jobs |

Start with the simplest representation that preserves meaning. Introduce a class when named state,
invariants, behavior, or a stable public interface repay the maintenance cost. Prefer composition
over inheritance unless the subtype relationship is genuine and substitutable.

## A repeatable design procedure

1. Write one sentence stating what the object represents—and what it does not represent.
2. List fields with types, units, valid domains, and missing-value policy.
3. State invariants that must hold immediately after construction.
4. Decide whether equal field values imply equal domain values.
5. Decide whether instances are snapshots or intentionally stateful.
6. Put intrinsic construction or state transitions in methods.
7. Put broader algorithms and policy in focused functions.
8. Document NumPy-style `Parameters`, semantics, units, `Returns`, and caller-relevant `Raises`.
9. Test ordinary values, boundaries, wrong types, invalid domains, and interactions.
10. Revisit the scientific assumptions separately from code correctness.

## Debugging object models

When an object or calculation surprises you, diagnose it in this order:

1. **Read the traceback from the final line upward.** Record the exception type and message.
2. **Inspect the runtime type.** Use `type(value)` and `isinstance`, not appearance alone.
3. **Inspect public state.** Print the object representation or selected documented attributes.
4. **Reproduce construction minimally.** Can the smallest valid object be created?
5. **Test the boundary.** Try a documented boundary and one value just outside it.
6. **Separate representation from calculation.** Did construction fail, or did a later function?
7. **Check units and assumptions.** Passing Python validation does not establish domain validity.
8. **Restart and run all.** Hidden notebook state can retain an older class definition.

Redefining a class in a notebook creates a new class object. Instances created before redefinition do
not silently change type. Restarting the kernel is the cleanest remedy while developing a class.

### Deliberate failures without breaking the notebook

Examples catch only the expected exception so the notebook can execute in CI. Broadly catching
`Exception` would also hide programming bugs. Inspect the messages: each points to the invalid
parameter and expected domain.

In [ ]:
failure_cases = (
    (lambda: Vector2D(float("nan"), 0.0), ValueError),
    (lambda: ParticleState("probe", 0.0, Vector2D(0, 0), Vector2D(0, 0)), ValueError),
    (lambda: moles_from_mass(water, mass_g=-1.0), ValueError),
    (lambda: ServiceStation("inspection", True), TypeError),
)

for operation, expected_exception in failure_cases:
    try:
        operation()
    except expected_exception as error:
        print(type(error).__name__ + ":", error)
    else:
        raise AssertionError(f"expected {expected_exception.__name__}")

## Common failure modes

| Failure | Why it happens | Better response |
| --- | --- | --- |
| constructor accepts nonsense | annotations are not runtime checks | validate public boundaries |
| derived field becomes stale | stored result no longer matches inputs | compute it or make a new snapshot |
| every function becomes a method | one class absorbs unrelated policy | use focused external functions |
| inheritance models “uses” or “has” | subtype relationship is false | prefer composition |
| unitless field names | compatible Python types hide incompatible quantities | name and document units |
| frozen object contains a list | freezing is shallow | use immutable nested values or document mutation |
| two instances share a list | mutable default was reused | use `default_factory` |
| old instance behaves differently | notebook class was redefined | restart kernel and run all |
| tests pass but recommendation is wrong | scientific assumptions were never tested | validate model choice with evidence |

## Guided practice: model a closed mathematical interval

Represent the closed interval `[lower, upper]`.

1. Use a frozen, slotted dataclass.
2. Normalize both endpoints with `_finite_float`.
3. Reject `lower > upper` with an actionable `ValueError`.
4. Expose `width` as a read-only property.
5. Implement `contains(value)` because membership is intrinsic interval behavior.
6. Check a degenerate interval, both endpoints, and an outside value.

First write the invariants in words. Then complete your version before comparing it with the
reference implementation below.

In [ ]:
@dataclass(frozen=True, slots=True)
class ClosedInterval:
    """Represent a finite closed interval on the real line.

    Parameters
    ----------
    lower : float
        Included lower endpoint.
    upper : float
        Included upper endpoint, no smaller than `lower`.

    Raises
    ------
    TypeError
        If an endpoint is not a real number.
    ValueError
        If an endpoint is not finite or `lower` exceeds `upper`.
    """

    lower: float
    upper: float

    def __post_init__(self) -> None:
        finite_lower = _finite_float(self.lower, parameter="lower")
        finite_upper = _finite_float(self.upper, parameter="upper")
        if finite_lower > finite_upper:
            raise ValueError(
                f"lower must not exceed upper; received {finite_lower} > {finite_upper}"
            )
        object.__setattr__(self, "lower", finite_lower)
        object.__setattr__(self, "upper", finite_upper)

    @property
    def width(self) -> float:
        """Distance between the endpoints."""

        return self.upper - self.lower

    def contains(self, value: object) -> bool:
        """Return whether a finite value lies in this closed interval.

        Parameters
        ----------
        value : object
            Candidate real value.

        Returns
        -------
        bool
            `True` when `lower <= value <= upper`.

        Raises
        ------
        TypeError
            If `value` is not a real number.
        ValueError
            If `value` is not finite.
        """

        finite_value = _finite_float(value, parameter="value")
        return self.lower <= finite_value <= self.upper


unit_interval = ClosedInterval(0, 1)
single_point = ClosedInterval(2, 2)

assert unit_interval.width == 1.0
assert unit_interval.contains(0.0)
assert unit_interval.contains(1.0)
assert not unit_interval.contains(1.01)
assert single_point.width == 0.0

### Guided reflection

- Why is `contains` a defensible method while `kinetic_energy` is a function?
- Why is a zero-width interval valid?
- Which facts are expressed by annotations, and which require runtime checks?
- Would silently swapping reversed endpoints be helpful normalization or dangerous error hiding?

**Success criterion:** your answers refer to the model's meaning, not only syntax.

## Independent practice: represent a prepared solution sample

Design an immutable `SolutionSample` with:

- a nonblank `sample_id`;
- a `solute: ChemicalSpecies`;
- nonnegative `solute_mass_g`;
- strictly positive `volume_liters`.

Then write a separate `molar_concentration(sample)` function returning moles per liter. Include
NumPy-style documentation, complete type hints, runtime validation, and specific exceptions.

**Success criteria:** a 5.844 g NaCl sample in 1.0 L is approximately 0.1 mol/L when its trusted
molar mass is 58.44 g/mol; zero solute is valid; zero volume, negative mass, blank identifiers, and
the wrong solute type are rejected. Add assertions for all five behaviors.

In [ ]:
# Reference solution: compare design decisions after attempting your own version.
@dataclass(frozen=True, slots=True)
class SolutionSample:
    """Represent a prepared solution sample.

    Parameters
    ----------
    sample_id : str
        Nonblank sample identifier.
    solute : ChemicalSpecies
        Trusted species dissolved in the sample.
    solute_mass_g : float
        Nonnegative solute mass in grams.
    volume_liters : float
        Strictly positive final solution volume in liters.

    Raises
    ------
    TypeError
        If field runtime types are incompatible with the model.
    ValueError
        If text is blank or a numeric field is outside its valid domain.
    """

    sample_id: str
    solute: ChemicalSpecies
    solute_mass_g: float
    volume_liters: float

    def __post_init__(self) -> None:
        if not isinstance(self.sample_id, str):
            raise TypeError(f"sample_id must be a string; received {self.sample_id!r}")
        normalized_id = self.sample_id.strip()
        if not normalized_id:
            raise ValueError(
                "sample_id must contain at least one non-whitespace character"
            )
        if not isinstance(self.solute, ChemicalSpecies):
            raise TypeError("solute must be a ChemicalSpecies instance")
        solute_mass = _finite_float(self.solute_mass_g, parameter="solute_mass_g")
        volume = _finite_float(self.volume_liters, parameter="volume_liters")
        if solute_mass < 0.0:
            raise ValueError(
                f"solute_mass_g must be nonnegative; received {self.solute_mass_g!r}"
            )
        if volume <= 0.0:
            raise ValueError(
                f"volume_liters must be positive; received {self.volume_liters!r}"
            )
        object.__setattr__(self, "sample_id", normalized_id)
        object.__setattr__(self, "solute_mass_g", solute_mass)
        object.__setattr__(self, "volume_liters", volume)


def molar_concentration(sample: SolutionSample) -> float:
    """Compute solute amount concentration.

    Parameters
    ----------
    sample : SolutionSample
        Validated sample with grams, grams per mole, and liters.

    Returns
    -------
    float
        Solute concentration in moles per liter.
    """

    amount_mol = moles_from_mass(sample.solute, mass_g=sample.solute_mass_g)
    return amount_mol / sample.volume_liters


sodium_chloride = ChemicalSpecies("sodium chloride", "NaCl", 58.44)
saline_sample = SolutionSample("salt-001", sodium_chloride, 5.844, 1.0)

assert abs(molar_concentration(saline_sample) - 0.1) < 1e-12
assert molar_concentration(SolutionSample("blank", sodium_chloride, 0, 1)) == 0.0

for invalid_construction, expected_exception in (
    (lambda: SolutionSample("zero-volume", sodium_chloride, 1, 0), ValueError),
    (lambda: SolutionSample("negative-mass", sodium_chloride, -1, 1), ValueError),
    (lambda: SolutionSample("   ", sodium_chloride, 1, 1), ValueError),
    (lambda: SolutionSample("wrong-solute", "NaCl", 1, 1), TypeError),
):
    try:
        invalid_construction()
    except expected_exception:
        pass
    else:
        raise AssertionError(f"expected {expected_exception.__name__}")

## Extension: choose a domain and defend the model

Choose one object:

- **mathematics:** a polynomial with immutable coefficient order;
- **physics:** a spring-mass state with positive mass and spring constant;
- **chemistry:** a reaction record that preserves reactants, products, and provenance;
- **industrial engineering:** a processing route composed of service stations.

Write the representation sentence, fields, units, invariants, and mutation policy before writing
code. Then implement one class and one separate function that computes something scientifically or
operationally meaningful.

**Success criteria:** the model cannot be constructed in at least two invalid states; public
interfaces have complete annotations and NumPy-style docstrings; exception messages are actionable;
at least five assertions cover a normal case, boundary, wrong type, invalid domain, and one scientific
invariant. In a short paragraph, defend one method-versus-function decision.

## Extension: resist premature abstraction

Compare two designs for a shared `ScientificObject` base class across vectors, particles, species,
and stations. What behavior could the base class truthfully promise? If the only answer is “has some
fields,” inheritance adds coupling without meaningful substitutability.

Try composition or a small protocol only after multiple real callers need a shared operation.
Generalization based on imagined reuse often produces an awkward interface. This is a professional
judgment exercise: more abstraction is not automatically more engineered.

## Retrieval practice

Answer without running code:

1. What is the relationship among a class, an instance, and `self`?
2. Why can an annotated constructor still accept a value of the wrong type?
3. What does `@dataclass` generate, and what scientific work does it not do?
4. What is an invariant? Name one invariant from each domain example.
5. Why are `first_vector == equivalent_vector` and `first_vector is equivalent_vector` different?
6. Why does `ParticleState` compose `Vector2D` rather than inherit from it?
7. When is a separate function preferable to a method?
8. Why does `ExperimentLog.notes` use `default_factory`?
9. Why is a frozen dataclass only shallowly immutable?
10. What can tests establish about `analyze_mm1`, and what can they not establish?

## Takeaway

A useful class is a compact claim about a domain object. Named fields preserve meaning; construction
enforces invariants; docstrings explain units and assumptions; annotations support tools; exceptions
make failure actionable; and tests keep the promise executable.

Use immutable dataclasses for value-like snapshots, ordinary classes for controlled lifecycles, and
composition for “has-a” relationships. Keep broader algorithms in typed functions when doing so
makes models smaller and computations easier to test. Neither a class nor a green test replaces
scientific judgment about whether the representation matches reality.

## Connection to the next notebook

Objects in memory disappear when the process ends. The next notebook reads text, CSV, JSON, and
JSON Lines with the standard library. That creates the next boundary problem: preserve the raw
record, parse it, construct a validated domain object, and report rejected records without silently
losing evidence.

Later, Lecture 3 moves reusable models and functions from notebooks into the `rice_dsm` package,
where tests, versioning, and continuous integration protect their public contracts.

## Further reading

- [Python tutorial: classes](https://docs.python.org/3/tutorial/classes.html)
- [Python standard library: `dataclasses`](https://docs.python.org/3/library/dataclasses.html)
- [Python data model](https://docs.python.org/3/reference/datamodel.html)
- [Python typing specification](https://typing.python.org/en/latest/spec/)
- [PEP 257: docstring conventions](https://peps.python.org/pep-0257/)
- [numpydoc style guide: documenting classes](https://numpydoc.readthedocs.io/en/latest/format.html#documenting-classes)

Documentation is a reference, not a substitute for testing your model's domain-specific invariants.